# Smart Recycling Assistant Model Training

This notebook trains a YOLO object detection model to detect:

1. Glass
2. Metal
3. Paper
4. Plastic
5. Waste

### Install the packages

In [12]:
%pip install -U ultralytics

Note: you may need to restart the kernel to use updated packages.


After this finishes, restart the notebook kernel if VS Code asks you to.

### Import libraries

In [13]:
from pathlib import Path

import torch
from ultralytics import YOLO

### Check which GPU is available

In [14]:
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    DEVICE = 0
elif torch.backends.mps.is_available():
    print("Apple Silicon GPU available")
    DEVICE = "mps"
else:
    print("No supported GPU detected. Training will use the CPU.")
    DEVICE = "cpu"

print("Selected device:", DEVICE)

PyTorch version: 2.13.0
CUDA available: False
Apple Silicon GPU available
Selected device: mps


### Set the project paths

Works whether this notebook sits in the repo root or in a `notebooks/` subfolder.

In [15]:
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent  # step up so DATA_DIR is always correct

DATA_DIR = PROJECT_ROOT / "data"
DATA_YAML = DATA_DIR / "data.yaml"

print("Project root:", PROJECT_ROOT)
print("Dataset YAML:", DATA_YAML)
print("YAML exists:", DATA_YAML.exists())

Project root: /Users/hafsahnasir/Developer/SNU/project/eco-vision
Dataset YAML: /Users/hafsahnasir/Developer/SNU/project/eco-vision/data/data.yaml
YAML exists: True


### Read and display `data.yaml`

In [16]:
print(DATA_YAML.read_text())

# No `path:` key on purpose — Ultralytics then resolves these relative to THIS
# file's own folder (data/), so it works on macOS/Windows from any working dir.
train: train/images
val: valid/images
test: test/images

nc: 5

names:
  0: Glass
  1: Metal
  2: Paper
  3: Plastic
  4: Waste



### Check the dataset folders

In [17]:
splits = ["train", "valid", "test"]

for split in splits:
    image_dir = DATA_DIR / split / "images"
    label_dir = DATA_DIR / split / "labels"

    image_files = (
        list(image_dir.glob("*.jpg"))
        + list(image_dir.glob("*.jpeg"))
        + list(image_dir.glob("*.png"))
    )
    label_files = list(label_dir.glob("*.txt"))

    print(f"\n{split.upper()}")
    print("Images:", len(image_files))
    print("Labels:", len(label_files))


TRAIN
Images: 3502
Labels: 3502

VALID
Images: 580
Labels: 580

TEST
Images: 45
Labels: 45


### Load a pretrained model

Use a small model first so training is faster. `n` is the smallest (nano) variant.

In [18]:
model = YOLO("yolo11n.pt")

### Train the model

In [20]:
import time

# --- Live ETA for the WHOLE run (Ultralytics' bar only shows per-epoch time) ---
def _on_train_start(trainer):
    trainer._t0 = time.time()

def _on_fit_epoch_end(trainer):
    done, total = trainer.epoch + 1, trainer.epochs
    elapsed = time.time() - trainer._t0
    per = elapsed / done
    left = per * (total - done)
    print(
        f"⏱  epoch {done}/{total} | elapsed {elapsed/60:.1f} min "
        f"| ~{left/60:.1f} min left | {per:.0f}s/epoch"
    )

# Guard so re-running this cell (without restarting the kernel) doesn't stack duplicates
if not getattr(model, "_eta_added", False):
    model.add_callback("on_train_start", _on_train_start)
    model.add_callback("on_fit_epoch_end", _on_fit_epoch_end)
    model._eta_added = True

# --- Faster settings for laptop training ---
#   imgsz : 640 (most accurate) -> 512 (balanced, here) -> 416 (fastest). Lower = faster.
#   batch : bigger = better GPU use. Drop back to 8 if you hit an out-of-memory error.
#   cache : keep images in RAM so each epoch skips re-reading them from disk.
training_results = model.train(
    data=str(DATA_YAML),
    epochs=30,
    imgsz=416,
    batch=16,
    cache=True,
    device=DEVICE,
    project=str(PROJECT_ROOT / "runs"),
    name="recycling_baseline",
    exist_ok=True,
    patience=15,
    pretrained=True,
    plots=True,
)

Ultralytics 8.4.95 🚀 Python-3.11.9 torch-2.13.0 MPS (Apple M2 Pro)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/Users/hafsahnasir/Developer/SNU/project/eco-vision/data/data.yaml, degrees=0.0, deterministic=True, device=mps, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=416, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=recycling_baseline, nbs=64, nms=False, opset=None, optimize=Fal

/Users/hafsahnasir/Developer/SNU/project/eco-vision/.venv/lib/python3.11/site-packages/torch/autograd/graph.py:979: UserWarning: index_put_with_accumulate_mps does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /Users/runner/work/pytorch/pytorch/aten/src/ATen/Context.cpp:190.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


       1/30      3.32G      1.263      2.495      1.123         43        416: 100% ━━━━━━━━━━━━ 219/219 2.6it/s 1:240.5sss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 19/19 1.9it/s 10.1s.5s
                   all        580       1319      0.355      0.184       0.12      0.078
⏱  epoch 1/30 | elapsed 1.6 min | ~47.4 min left | 98s/epoch

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


/Users/hafsahnasir/Developer/SNU/project/eco-vision/.venv/lib/python3.11/site-packages/torch/autograd/graph.py:979: UserWarning: index_put_with_accumulate_mps does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /Users/runner/work/pytorch/pytorch/aten/src/ATen/Context.cpp:190.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


       2/30      3.32G      1.353      2.276       1.16         53        416: 100% ━━━━━━━━━━━━ 219/219 2.8it/s 1:190.4sss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 19/19 2.2it/s 8.6s0.4s
                   all        580       1319      0.355      0.165      0.107     0.0698
⏱  epoch 2/30 | elapsed 3.1 min | ~44.1 min left | 94s/epoch

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


...

      28/30      3.35G       1.05      1.153     0.9808         33        416: 100% ━━━━━━━━━━━━ 219/219 3.1it/s 1:110.4sss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 19/19 1.9it/s 9.8s0.5s
                   all        580       1319      0.436       0.21      0.171      0.124
⏱  epoch 28/30 | elapsed 42.0 min | ~3.0 min left | 90s/epoch

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      29/30      3.34G       1.19       1.23     0.9733         39        416: 0% ──────────── 0/219  1.0s

/Users/hafsahnasir/Developer/SNU/project/eco-vision/.venv/lib/python3.11/site-packages/torch/autograd/graph.py:979: UserWarning: index_put_with_accumulate_mps does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /Users/runner/work/pytorch/pytorch/aten/src/ATen/Context.cpp:190.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      29/30      3.35G      1.034      1.134     0.9738         27        416: 100% ━━━━━━━━━━━━ 219/219 3.2it/s 1:090.4sss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 19/19 2.1it/s 9.2s0.5s
                   all        580       1319      0.524      0.197       0.19       0.14
⏱  epoch 29/30 | elapsed 43.4 min | ~1.5 min left | 90s/epoch

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      30/30       3.3G      1.119      1.179     0.9714         51        416: 0% ──────────── 0/219  1.0s

/Users/hafsahnasir/Developer/SNU/project/eco-vision/.venv/lib/python3.11/site-packages/torch/autograd/graph.py:979: UserWarning: index_put_with_accumulate_mps does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /Users/runner/work/pytorch/pytorch/aten/src/ATen/Context.cpp:190.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      30/30      3.36G      1.028      1.094     0.9743         44        416: 100% ━━━━━━━━━━━━ 219/219 3.2it/s 1:090.4sss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 19/19 2.0it/s 9.3s0.5s
                   all        580       1319      0.457      0.223      0.184      0.136
⏱  epoch 30/30 | elapsed 44.7 min | ~0.0 min left | 89s/epoch

30 epochs completed in 0.745 hours.
Optimizer stripped from /Users/hafsahnasir/Developer/SNU/project/eco-vision/runs/recycling_baseline/weights/last.pt, 5.4MB
Optimizer stripped from /Users/hafsahnasir/Developer/SNU/project/eco-vision/runs/recycling_baseline/weights/best.pt, 5.4MB

Validating /Users/hafsahnasir/Developer/SNU/project/eco-vision/runs/recycling_baseline/weights/best.pt...
Ultralytics 8.4.95 🚀 Python-3.11.9 torch-2.13.0 MPS (Apple M2 Pro)
YOLO11n summary (fused): 101 layers, 2,583,127 parameters, 0 gradients, 6.3 GFLOPs
                 Class     Images  Instances      Box(

In [ ]:
# If needed, continue training for 20 additional epochs using the best learned weights
from ultralytics import YOLO

BEST_WEIGHTS = (
    PROJECT_ROOT
    / "runs"
    / "recycling_baseline"
    / "weights"
    / "best.pt"
)

continued_model = YOLO(str(BEST_WEIGHTS))

continued_results = continued_model.train(
    data=str(DATA_YAML),
    epochs=20,
    imgsz=416,
    batch=16,
    cache=True,
    device=DEVICE,
    project=str(PROJECT_ROOT / "runs"),
    name="recycling_extra_20",
    exist_ok=True,
    patience=10,
    pretrained=True,
    plots=True,
)

Ultralytics 8.4.95 🚀 Python-3.11.9 torch-2.13.0 MPS (Apple M2 Pro)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/Users/hafsahnasir/Developer/SNU/project/eco-vision/data/data.yaml, degrees=0.0, deterministic=True, device=mps, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=20, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=416, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/Users/hafsahnasir/Developer/SNU/project/eco-vision/runs/recycling_baseline/weights/best.pt, momentum=0.937, mosaic=1.0, mult

/Users/hafsahnasir/Developer/SNU/project/eco-vision/.venv/lib/python3.11/site-packages/torch/autograd/graph.py:979: UserWarning: index_put_with_accumulate_mps does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /Users/runner/work/pytorch/pytorch/aten/src/ATen/Context.cpp:190.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


       1/20      3.42G      1.052      1.324      1.017         43        416: 100% ━━━━━━━━━━━━ 219/219 2.6it/s 1:230.4sss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 19/19 2.0it/s 9.6s0.5s
                   all        580       1319      0.429      0.209      0.161      0.119

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


### Show where the results were saved

In [21]:
RUN_DIR = PROJECT_ROOT / "runs" / "recycling_baseline"

print("Training output:", RUN_DIR)
print("Best model:", RUN_DIR / "weights" / "best.pt")
print("Last model:", RUN_DIR / "weights" / "last.pt")

Training output: /Users/hafsahnasir/Developer/SNU/project/eco-vision/runs/recycling_baseline
Best model: /Users/hafsahnasir/Developer/SNU/project/eco-vision/runs/recycling_baseline/weights/best.pt
Last model: /Users/hafsahnasir/Developer/SNU/project/eco-vision/runs/recycling_baseline/weights/last.pt


### Load the best model

In [22]:
BEST_MODEL_PATH = RUN_DIR / "weights" / "best.pt"

assert BEST_MODEL_PATH.exists(), "The best model file was not found."

best_model = YOLO(str(BEST_MODEL_PATH))

print("Loaded:", BEST_MODEL_PATH)

Loaded: /Users/hafsahnasir/Developer/SNU/project/eco-vision/runs/recycling_baseline/weights/best.pt


### Evaluate it on the validation set

In [23]:
validation_metrics = best_model.val(
    data=str(DATA_YAML),
    split="val",
    device=DEVICE,
    plots=True,
)

print("mAP50:", validation_metrics.box.map50)
print("mAP50-95:", validation_metrics.box.map)
print("Precision:", validation_metrics.box.mp)
print("Recall:", validation_metrics.box.mr)

Ultralytics 8.4.95 🚀 Python-3.11.9 torch-2.13.0 MPS (Apple M2 Pro)
YOLO11n summary (fused): 101 layers, 2,583,127 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 88.3±53.4 MB/s, size: 29.8 KB)
val: Scanning /Users/hafsahnasir/Developer/SNU/project/eco-vision/data/valid/labels.cache... 580 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 580/580 486.5Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 37/37 3.9it/s 9.6s0.2s
                   all        580       1319      0.437      0.245      0.192      0.142
                 Glass         51         69          1          0      0.038     0.0302
                 Metal        123        209      0.315       0.34      0.215      0.169
                 Paper        142        267      0.278      0.223      0.154       0.11
               Plastic        393        633      0.463      0.548      0.477      0.347
                 Was

### Evaluate it on the test set

In [29]:
test_metrics = best_model.val(
    data=str(DATA_YAML),
    split="test",
    device=DEVICE,
    plots=True,
)

print("Test mAP50:", test_metrics.box.map50)
print("Test mAP50-95:", test_metrics.box.map)
print("Test precision:", test_metrics.box.mp)
print("Test recall:", test_metrics.box.mr)

Ultralytics 8.4.95 🚀 Python-3.11.9 torch-2.13.0 MPS (Apple M2 Pro)
val: Fast image access ✅ (ping: 0.1±0.1 ms, read: 64.0±61.4 MB/s, size: 46.4 KB)
val: Scanning /Users/hafsahnasir/Developer/SNU/project/eco-vision/data/test/labels.cache... 45 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 45/45 7.5Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 3/3 3.5it/s 0.9s0.6s
                   all         45        159      0.458      0.247      0.203      0.162
                 Glass          1         26          1          0          0          0
                 Metal          6         13      0.324      0.462      0.313      0.273
                 Paper         15         28      0.222      0.179      0.168      0.125
               Plastic         33         70      0.448      0.414      0.403      0.313
                 Waste         11         22      0.296      0.182      0.132      0.101
Speed: 0.1ms preproces

### Run predictions on test images

In [25]:
TEST_IMAGES = DATA_DIR / "test" / "images"

prediction_results = best_model.predict(
    source=str(TEST_IMAGES),
    conf=0.50,
    imgsz=640,
    device=DEVICE,
    save=True,
    project=str(PROJECT_ROOT / "runs"),
    name="test_predictions",
    exist_ok=True,
)


image 1/45 /Users/hafsahnasir/Developer/SNU/project/eco-vision/data/test/images/000000_jpg.rf.0cca9fadb7533c08488f934280a8ade3.jpg: 640x640 1 Plastic, 14.2ms
image 2/45 /Users/hafsahnasir/Developer/SNU/project/eco-vision/data/test/images/000001_jpg.rf.e20ac5d46de710cf33ec7dbb23e27a1b.jpg: 640x640 2 Plastics, 14.9ms
image 3/45 /Users/hafsahnasir/Developer/SNU/project/eco-vision/data/test/images/000002_jpg.rf.8d166b944a323ca91e2c2b09c7c8c68c.jpg: 640x640 1 Plastic, 15.6ms
image 4/45 /Users/hafsahnasir/Developer/SNU/project/eco-vision/data/test/images/000005_jpg.rf.9d95154c7b57eaf429f940ebf050df8b.jpg: 640x640 2 Plastics, 10.2ms
image 5/45 /Users/hafsahnasir/Developer/SNU/project/eco-vision/data/test/images/000006_jpg.rf.2936c2a285ae461b1a0352c11d3f714e.jpg: 640x640 (no detections), 11.2ms
image 6/45 /Users/hafsahnasir/Developer/SNU/project/eco-vision/data/test/images/000006_jpg.rf.c9c8e06822d9a2e6091bea0fb3259b11.jpg: 640x640 1 Plastic, 10.1ms
image 7/45 /Users/hafsahnasir/Developer/SNU

### Display a prediction inside the notebook

In [26]:
import matplotlib.pyplot as plt

if prediction_results:
    predicted_image = prediction_results[0].plot()

    plt.figure(figsize=(10, 8))
    plt.imshow(predicted_image[..., ::-1])
    plt.axis("off")
    plt.show()

<Figure size 1000x800 with 1 Axes>

### Copy the final model into a `models` folder

In [27]:
import shutil

MODELS_DIR = PROJECT_ROOT / "models"
MODELS_DIR.mkdir(exist_ok=True)

FINAL_MODEL_PATH = MODELS_DIR / "recycling_model.pt"

shutil.copy2(BEST_MODEL_PATH, FINAL_MODEL_PATH)

print("Final model copied to:", FINAL_MODEL_PATH)

Final model copied to: /Users/hafsahnasir/Developer/SNU/project/eco-vision/models/recycling_model.pt
